# WINGS3 — Autolog tracing

Run in JupyterLab workbench **wings3-demo**, project `my-first-model`.

**Red thread:** without traces you cannot debug the agent. `mlflow.langchain.autolog()` is the one line that captures LLM calls and the calculator span.

**Llama 3.2 3B:** one tool call per turn. Extra queries often land as **Error** — that is the debug beat in the Traces UI, not a failure of the hour.

On stage, stop at each **SHOW:** comment. Run **one** query (`256 ÷ 16`).


## 0. Optional: git pull

JupyterLab root is this clone. Skip on stage if already current. Cluster must reach GitHub.


In [ ]:
# Optional: update from GitHub. Skip on stage if already current.
!git pull --ff-only


## 1b. Install deps if `langchain_core` is missing

Run this **before** the env cell on a fresh kernel (env imports langchain/mlflow). Skip if those imports already work. Re-run after a workbench restart — the venv is not on the PVC. `--extra-index-url` is required: the RHOAI 3.4 RHAI index has langgraph 1.x only and may not have `langchain-core`.


In [ ]:
# Kernel venv is not on the PVC. Skip if `import langchain_core` already works.
%pip install -r ../agent-tracing/requirements.txt --extra-index-url https://pypi.org/simple


## 1. Workbench env

Tracking URI is injected when the notebook has `opendatahub.io/mlflow-instance=mlflow`. You still set `MLFLOW_WORKSPACE`. Do **not** open `traced_agent.py` on stage.


In [ ]:
import importlib
import logging
import math
import os
import sys
import warnings
from pathlib import Path

import mlflow
from langchain_core.tools import tool

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

os.environ.setdefault("MLFLOW_WORKSPACE", "my-first-model")
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", "wings3-agent-tracing")
os.environ.setdefault("MAAS_API_KEY", "unused")
os.environ.setdefault("MAAS_MODEL", "llama-32-3b-instruct")
os.environ.setdefault(
    "MAAS_BASE_URL",
    "http://llama-32-3b-instruct-predictor.my-first-model.svc.cluster.local:8080/v1",
)

demo = Path("../agent-tracing").resolve()
if not (demo / "traced_agent.py").is_file():
    raise FileNotFoundError(f"expected traced_agent.py next to notebooks: {demo}")
sys.path.insert(0, str(demo))
traced_agent = importlib.import_module("traced_agent")
create_agent_graph = traced_agent.create_agent_graph
get_config_from_env = traced_agent.get_config_from_env

for k in (
    "MLFLOW_TRACKING_URI",
    "MLFLOW_WORKSPACE",
    "MLFLOW_K8S_INTEGRATION",
    "MLFLOW_TRACKING_AUTH",
):
    print(f"{k}={os.environ.get(k)}")


## 2. SHOW: calculator tool

This is the span autolog will capture. Without autolog you would miss it unless you wrapped it in a manual span.


In [ ]:
# SHOW: calculator-only — this cluster's 3B model supports one tool call per turn.
# b is optional: Llama 3.2 3B omits it on sqrt; a required b raises pydantic Field required.
@tool
def calculator(operation: str, a: float, b: float | None = None) -> str:
    """Arithmetic: add, subtract, multiply, divide, sqrt, power. For sqrt, pass only a."""
    ops = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y else "Error: divide by zero",
        "sqrt": lambda x, _: math.sqrt(x),
        "power": lambda x, y: x**y,
    }
    if operation not in ops:
        return f"Unknown operation {operation}"
    if operation != "sqrt" and b is None:
        return f"Error: {operation} needs two numbers a and b"
    result = ops[operation](a, b)
    if operation == "sqrt":
        return f"sqrt({a}) = {result}"
    return f"Result: {result}"


print("Tool:", calculator.name)


## 3. SHOW: `mlflow.langchain.autolog()`

Call this **before** the agent runs. No manual spans.


In [ ]:
uri = os.environ.get("MLFLOW_TRACKING_URI")
if not uri:
    raise RuntimeError("MLFLOW_TRACKING_URI is not set")
if not os.environ.get("MLFLOW_WORKSPACE"):
    raise RuntimeError("Set MLFLOW_WORKSPACE=my-first-model")

experiment = os.environ.get("MLFLOW_EXPERIMENT_NAME", "wings3-agent-tracing")
mlflow.set_tracking_uri(uri)
mlflow.set_experiment(experiment)

# SHOW: this is the instrumentation — one line, no manual spans
mlflow.langchain.autolog()

config = get_config_from_env()
agent = create_agent_graph(config, tools=[calculator])
print(f"MLflow: {uri}")
print(f"Workspace: {os.environ['MLFLOW_WORKSPACE']}")
print(f"Experiment: {experiment}")
print(f"Model: {config.model}")


## 4. SHOW: one query on stage

`Calculate 256 divided by 16` should print `16.0`. Do **not** run extra queries on camera (~4 minutes; 3B often Errors).


In [ ]:
# SHOW: one live query — 256 ÷ 16. Expected: 16.0
query = "Calculate 256 divided by 16"
print(f"--- Query: {query}")
result = agent.invoke({"messages": [{"role": "user", "content": query}]})
print(result["messages"][-1].content)
print("Done → MLflow UI → Traces → Details & Timeline")


## 5. Optional extra queries (skip on stage)

Leave this cell unrun unless you are rehearsing Error traces. A rehearsal Error is enough for the debug beat.


In [ ]:
# Skip on stage. Extra queries often Error on Llama 3.2 3B.
EXTRA = [
    "What is 25 times 17?",
    "What is the square root of 144?",
]
for query in EXTRA:
    print(f"\n--- Query: {query}")
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": query}]})
        print(result["messages"][-1].content)
    except Exception as exc:
        print(f"Error: {exc}")


## 6. Open traces — debug an Error, then the OK tree

Use the **standalone** `/mlflow` UI, not the embedded Experiments view.

1. Workspace **my-first-model** → experiment **wings3-agent-tracing** → **Traces**.
2. Open an **Error** row (rehearsal Error is fine if this live query is OK). Name the failure. Do not linger.
3. Open the **OK** row **Calculate 256 divided by 16**. Do not pick latest by default.
4. **Details & Timeline:** LangGraph → ChatOpenAI → **calculator** → ChatOpenAI.

If the drawer does not open, add `selectedEvaluationId=<trace-id>` to the Traces URL.
